# 按轨迹分段查看时间冲突与GNSS

写入日期：2026-10-21；修改日期：2026-10-21。本次修改：将Markdown内逐条GNSS记录改为连续同城访问汇总，显示城市、最早进入、最晚离开及GNSS行数。原四个CSV保留、不覆盖；原始数据、接续、工时统计和第二步结果不改。

## 1 说明

正式输入仍是三份第一步年度候选接续，以及有冲突年份的只读GNSS。范围为2021—2023年各年5月1日至6月25日、全部轨迹城市。只要“终点进入时间早于起点离开时间”就算冲突，不因同城或多级排除；等于0不算。

按“年份+农机编号+链段编号”区分轨迹，当前链段均为1。同一农机不同年份分段，不跨年拼接。完整城市路径来自该农机当年全部输入记录，连续同城合并、离开后回访保留；不代表观测外或全年真实路线。

当前输出为Results/00_数据审计/冲突轨迹与GNSS.md。每段第一行显示年份、did、链段及完整城市路径，下面放did、链段编号、城市、最早进入时间、最晚离开时间、GNSS行数六列。按完整排序将连续同城记录合并为一次访问，进入取该次访问全部GNSS的最早start_time，离开取最晚end_time。离开后返回同城单列，不能将A→B→A的两次A合成一行。GNSS行数是每次访问包含的原始行数，不去重；全部访问的行数之和必须还原完整轨迹GNSS行数。两时点表示访问覆盖范围，不代表一直在工作。

路径、访问与完整原始字段仍在内存中用于核对，但不再生成四个明细CSV。旧格式CSV原样保留，其生成代码已备份，清单记录旧格式用途；本文件只供人工查看，不进入研究样本或链路。
2026-10-21修改：按人工删除后数据重查。支持没有时间冲突的正常结果，额外保存三年时间冲突检查CSV；旧四份CSV标记为删除前历史结果，不再作为当前原行位置。


## 2 参数

依赖pandas、numpy、duckdb。默认拒绝覆盖当前Markdown文件；明确重跑时设置CONFLICT_TRAJECTORY_ALLOW_OVERWRITE=1。普通重跑不自动归档。旧CSV不随当前Notebook重新生成。

In [1]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

project_root = next((path for path in [Path.cwd(), *Path.cwd().parents]
                     if (path / 'README.md').is_file() and (path / 'Data').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('请从项目根目录或Notebook所在目录运行。')
years = [2021, 2022, 2023]
notebook_relative = 'Code/01_轨迹转城市接续/export_conflict_trajectories.ipynb'
result_dir = project_root / 'Results/00_数据审计'
report_path = result_dir / '冲突轨迹与GNSS.md'
legacy_generator_relative = 'Archive/2026-10-21_冲突轨迹分段排版前/Code__01_轨迹转城市接续__export_conflict_trajectories.ipynb'
allow_overwrite = os.getenv('CONFLICT_TRAJECTORY_ALLOW_OVERWRITE', '0') == '1'
if report_path.exists() and not allow_overwrite:
    raise FileExistsError('同名结果已存在；确认重跑时设置CONFLICT_TRAJECTORY_ALLOW_OVERWRITE=1。')
manifest_path = project_root / 'Data/data_manifest.csv'
manifest_table = pd.read_csv(manifest_path, encoding='utf-8-sig', dtype='string')
assert not manifest_table['relative_path'].duplicated().any()
manifest = manifest_table.set_index('relative_path')
source_hashes = {}

def file_fingerprint(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def verify_source(relative):
    assert relative in manifest.index and manifest.loc[relative, 'formal_status'] == '正式', relative
    path = project_root / relative
    assert path.stat().st_size == int(manifest.loc[relative, 'file_size_bytes']), relative
    source_hashes[relative] = file_fingerprint(path)
    assert source_hashes[relative] == manifest.loc[relative, 'sha256'], f'文件指纹不符：{relative}'
    return path

# 与第一步一致；原文件行号负责打破最后的排序并列，不对近重复记录去重。
sort_sql = 'date_value, start_ts, end_ts, center_ts NULLS LAST, info_id NULLS LAST, id NULLS LAST, source_row_no'


## 3 读取

逐年检查全部候选接续，按时间戳重新判断负间隔，再与已有小时数和时间冲突标记核对。按完整原文件编号后再选择农机，保证源数据行号不是筛选后的行号。

In [2]:
year_candidates = {}
conflict_parts = []
year_checks = []
for year in years:
    relative = f'Data/03_有向边数据/{year}年农机城市接续.csv'
    candidates = pd.read_csv(verify_source(relative), encoding='utf-8-sig', dtype='string', keep_default_na=False)
    assert len(candidates) == int(manifest.loc[relative, 'row_count'])
    required = ['边记录编号', '农机编号', '年份', '链段编号', '起点访问序号', '终点访问序号',
                '起点城市', '终点城市', '起点离开时间', '终点进入时间', '时间间隔小时', '是否时间冲突']
    assert candidates[required].apply(lambda column: column.str.strip().ne('')).all().all()
    assert candidates['年份'].eq(str(year)).all() and candidates['链段编号'].eq('1').all()
    assert not candidates['边记录编号'].duplicated().any()
    gap_seconds = (pd.to_datetime(candidates['终点进入时间'], errors='raise')
                   - pd.to_datetime(candidates['起点离开时间'], errors='raise')).dt.total_seconds()
    assert np.isfinite(gap_seconds).all()
    assert np.allclose(gap_seconds / 3600, pd.to_numeric(candidates['时间间隔小时']), rtol=0, atol=1e-9)
    has_time_conflict = gap_seconds.lt(0)
    assert candidates['是否时间冲突'].str.lower().isin(['true', 'false']).all()
    assert candidates['是否时间冲突'].str.lower().eq('true').eq(has_time_conflict).all()
    candidates.insert(0, '轨迹编号', candidates['年份'] + '-' + candidates['农机编号'] + '-' + candidates['链段编号'])
    candidates['接续来源文件'] = relative
    year_candidates[year] = candidates
    conflicts = candidates.loc[has_time_conflict].copy()
    conflict_parts.append(conflicts)
    year_checks.append({'年份': year, '候选接续数': len(candidates), '冲突接续数': len(conflicts),
                        '冲突轨迹数': conflicts['轨迹编号'].nunique()})
conflict_table = pd.concat(conflict_parts, ignore_index=True)
# 没有冲突也是有效结果，继续输出检查说明。
display(pd.DataFrame(year_checks))


,年份,候选接续数,冲突接续数,冲突轨迹数
0,2021,110458,0,0
1,2022,274939,0,0
2,2023,500068,0,0


## 4 处理

从有冲突的农机读取当年全部作业段，按第一步方法重建完整访问。每条候选接续的城市、时间、访问位置、原始记录数、首末行号都要与重建结果一致；核对不通过即停止。

GNSS表每条原始行只列一次，不按它涉及的冲突次数重复。冲突起点、冲突终点编号指出整次访问涉及哪些冲突；“提供判断时间接续编号”进一步指出该原始行确实提供了起点最晚结束或终点最早开始时间。空白表示不涉及该项，不代表缺失时间被填补。

In [3]:
path_records, visit_records, raw_parts = [], [], []
connection = duckdb.connect()
connection.execute('SET preserve_insertion_order=true')
try:
    for year in years:
        conflicts = conflict_table.loc[conflict_table['年份'].eq(str(year))]
        if conflicts.empty:
            continue
        raw_relative = f'Data/00_原始数据_只读/final_filtered_{str(year)[-2:]}_modified.csv'
        raw_path = verify_source(raw_relative)
        connection.register('wanted', pd.DataFrame({'did': conflicts['农机编号'].unique()}))
        # 完整CSV先编号、核对总行数，再选择轨迹；只扫描对应冲突年份。
        # 用DuckDB顺序读取，避免把数百万行原始数据全部放进Python内存。
        connection.execute("""
            CREATE OR REPLACE TEMP VIEW raw_source AS
            SELECT row_number() OVER ()::BIGINT AS source_row_no, *
            FROM read_csv($raw_path, header=true, all_varchar=true, parallel=false, ignore_errors=false)
        """.replace('$raw_path', "'" + str(raw_path).replace("'", "''") + "'"))
        total_rows = connection.execute('SELECT count(*) FROM raw_source').fetchone()[0]
        assert total_rows == int(manifest.loc[raw_relative, 'row_count'])
        raw = connection.execute('SELECT * FROM raw_source WHERE trim(did) IN (SELECT did FROM wanted)').df()
        raw_columns = [column for column in raw.columns if column != 'source_row_no']
        assert {'did', 'date', 'start_time', 'end_time', 'center_time', 'center_location', 'info_id', 'id'}.issubset(raw_columns)
        connection.register('selected_raw', raw)
        connection.execute("""
            CREATE OR REPLACE TEMP VIEW parsed AS
            SELECT source_row_no, NULLIF(trim(info_id),'') AS info_id, NULLIF(trim(id),'') AS id,
              trim(did) AS did, CAST(trim(date) AS DATE) AS date_value,
              CAST(trim(start_time) AS TIMESTAMP) AS start_ts, CAST(trim(end_time) AS TIMESTAMP) AS end_ts,
              COALESCE(try_strptime(trim(center_time),'%Y%m%d%H%M%S'),
                       try_cast(trim(center_time) AS TIMESTAMP)) AS center_ts,
              NULLIF(trim(center_location),'') AS city
            FROM selected_raw
        """)
        parsed = connection.execute(f"""
            SELECT *,row_number() OVER(PARTITION BY did ORDER BY {sort_sql})::BIGINT AS track_row_order
            FROM parsed ORDER BY did, {sort_sql}
        """).df()
        assert not parsed[['did', 'date_value', 'start_ts', 'end_ts']].isna().any().any()
        assert parsed['end_ts'].ge(parsed['start_ts']).all()
        assert parsed['start_ts'].dt.year.eq(year).all() and parsed['end_ts'].dt.year.eq(year).all()
        assert parsed['date_value'].between(pd.Timestamp(year, 5, 1), pd.Timestamp(year, 6, 25)).all()
        connection.execute(f"""
            CREATE OR REPLACE TEMP VIEW flagged AS
            SELECT *, CASE WHEN lag(city) OVER(PARTITION BY did ORDER BY {sort_sql})
                                 IS DISTINCT FROM city THEN 1 ELSE 0 END AS new_visit
            FROM parsed WHERE city IS NOT NULL
        """)
        visit_rows = connection.execute(f"""
            SELECT *,sum(new_visit) OVER(PARTITION BY did ORDER BY {sort_sql}
                ROWS UNBOUNDED PRECEDING)::BIGINT AS visit_order FROM flagged
        """).df()
        connection.register('visit_rows', visit_rows)
        visits = connection.execute(f"""
            SELECT did, visit_order, first(city ORDER BY {sort_sql}) AS city,
              min(start_ts) AS entry_time, max(end_ts) AS exit_time, count(*) AS record_count,
              first(source_row_no ORDER BY {sort_sql}) AS first_row,
              last(source_row_no ORDER BY {sort_sql}) AS last_row
            FROM visit_rows GROUP BY 1,2 ORDER BY 1,2
        """).df().set_index(['did', 'visit_order'])
        endpoint_ids, basis_ids = {}, {}
        selected_candidates = year_candidates[year].loc[year_candidates[year]['轨迹编号'].isin(conflicts['轨迹编号'])]
        for _, edge in selected_candidates.iterrows():
            # 对有冲突的轨迹，检查全部候选边，不能只校验冲突两端。
            for prefix in ['起点', '终点']:
                key = (edge['农机编号'], int(edge[prefix + '访问序号']))
                visit = visits.loc[key]
                assert visit.city == edge[prefix + '城市']
                assert visit.entry_time == pd.Timestamp(edge[prefix + '进入时间'])
                assert visit.exit_time == pd.Timestamp(edge[prefix + '离开时间'])
                for field, value in [('原始记录数', visit.record_count), ('首行号', visit.first_row), ('末行号', visit.last_row)]:
                    assert int(edge[prefix + field]) == int(value)
                if float(edge['时间间隔小时']) < 0:
                    endpoint_ids.setdefault((*key, prefix), []).append(edge['边记录编号'])
                    members = visit_rows.loc[visit_rows.did.eq(key[0]) & visit_rows.visit_order.eq(key[1])]
                    mask = members.end_ts.eq(visit.exit_time) if prefix == '起点' else members.start_ts.eq(visit.entry_time)
                    assert mask.any()
                    for source_no in members.loc[mask, 'source_row_no']:
                        basis_ids.setdefault(int(source_no), []).append(prefix + ':' + edge['边记录编号'])
        for did in sorted(conflicts['农机编号'].unique()):
            trajectory_id = f'{year}-{did}-1'
            track_visits = visits.loc[did]
            track_raw = parsed.loc[parsed.did.eq(did)]
            track_edges = conflicts.loc[conflicts['农机编号'].eq(did)]
            assert track_visits.index.tolist() == list(range(1, len(track_visits) + 1))
            assert len(selected_candidates.loc[selected_candidates['农机编号'].eq(did)]) == len(track_visits) * (len(track_visits) - 1) // 2
            path_records.append({
                '轨迹编号': trajectory_id, '年份': year, '农机编号': did, '链段编号': 1,
                '完整城市路径': '→'.join(track_visits.city),
                '带序号城市路径': '→'.join(f'{order}:{visit.city}' for order, visit in track_visits.iterrows()),
                '城市访问数': len(track_visits), 'GNSS原始行数': len(track_raw),
                '城市缺失行数': int(track_raw.city.isna().sum()), '冲突接续数': len(track_edges),
                '冲突一级数': int(track_edges['边级'].eq('1').sum()),
                '冲突多级数': int(pd.to_numeric(track_edges['边级']).gt(1).sum()),
                '冲突接续编号': '；'.join(track_edges['边记录编号']),
                '接续来源文件': f'Data/03_有向边数据/{year}年农机城市接续.csv', 'GNSS来源文件': raw_relative})
            for order, visit in track_visits.iterrows():
                visit_records.append({
                    '轨迹编号': trajectory_id, '年份': year, '农机编号': did, '链段编号': 1,
                    '访问序号': int(order), '城市': visit.city, '进入时间': str(visit.entry_time),
                    '离开时间': str(visit.exit_time), '原始记录数': int(visit.record_count),
                    '首行号': int(visit.first_row), '末行号': int(visit.last_row),
                    '冲突起点接续编号': '；'.join(endpoint_ids.get((did, order, '起点'), [])),
                    '冲突终点接续编号': '；'.join(endpoint_ids.get((did, order, '终点'), []))})
        metadata = parsed[['source_row_no', 'did', 'track_row_order', 'city']].merge(
            visit_rows[['source_row_no', 'visit_order']], on='source_row_no', how='left', validate='one_to_one')
        metadata['visit_order'] = metadata['visit_order'].astype('Int64')
        metadata['轨迹编号'] = str(year) + '-' + metadata.did + '-1'
        metadata['年份'], metadata['农机编号'], metadata['链段编号'] = year, metadata.did, 1
        metadata['GNSS来源文件'] = raw_relative
        metadata['城市缺失未参与路径'] = metadata.city.isna()
        metadata['GNSS_CSV行号'] = metadata.source_row_no + 1
        for prefix in ['起点', '终点']:
            metadata['冲突' + prefix + '接续编号'] = [
                '；'.join(endpoint_ids.get((did, int(order), prefix), [])) if pd.notna(order) else ''
                for did, order in zip(metadata.did, metadata.visit_order)]
        metadata['提供判断时间接续编号'] = metadata.source_row_no.map(lambda value: '；'.join(basis_ids.get(int(value), [])))
        meta_columns = ['轨迹编号', '年份', '农机编号', '链段编号', 'track_row_order', 'visit_order',
                        '城市缺失未参与路径', 'source_row_no', 'GNSS_CSV行号', 'GNSS来源文件',
                        '冲突起点接续编号', '冲突终点接续编号', '提供判断时间接续编号']
        raw_output = metadata[meta_columns].merge(raw, on='source_row_no', how='left', validate='one_to_one')
        # 原始26列只加前缀，值不改；原始空值仍写为空白，不补0。
        pd.testing.assert_frame_equal(raw_output.set_index('source_row_no')[raw_columns].sort_index(),
                                      raw.set_index('source_row_no')[raw_columns].sort_index())
        raw_output = raw_output.rename(columns={'source_row_no': 'GNSS源数据行号',
            'track_row_order': '轨迹内行序号', 'visit_order': '访问序号',
            **{column: 'GNSS_' + column for column in raw_columns}})
        raw_parts.append(raw_output.sort_values(['轨迹编号', '轨迹内行序号'], ignore_index=True))
finally:
    connection.close()
path_columns = ['轨迹编号','年份','农机编号','链段编号','完整城市路径','带序号城市路径','城市访问数',
                'GNSS原始行数','城市缺失行数','冲突接续数','冲突一级数','冲突多级数',
                '冲突接续编号','接续来源文件','GNSS来源文件']
path_table = pd.DataFrame(path_records,columns=path_columns).sort_values(['年份','农机编号'],ignore_index=True)
visit_columns = ['轨迹编号','年份','农机编号','链段编号','访问序号','城市','进入时间','离开时间',
                 '原始记录数','首行号','末行号','冲突起点接续编号','冲突终点接续编号']
visit_table = pd.DataFrame(visit_records,columns=visit_columns).sort_values(['轨迹编号','访问序号'],ignore_index=True)
raw_table = (pd.concat(raw_parts,ignore_index=True) if raw_parts else
             pd.DataFrame(columns=['轨迹编号','GNSS来源文件','GNSS源数据行号','轨迹内行序号','访问序号']))


## 5 检查

核对每条冲突都能找到完整路径和两个端点；GNSS不漏行、不重复，所有访问与源行对应。路径内非冲突访问也必须保留。三年没有冲突的年份在年度检查表中显示0，不制造空轨迹。

In [4]:
assert not path_table['轨迹编号'].duplicated().any()
assert not visit_table[['轨迹编号', '访问序号']].duplicated().any()
assert not conflict_table['边记录编号'].duplicated().any()
assert not raw_table[['GNSS来源文件', 'GNSS源数据行号']].duplicated().any()
assert set(path_table['轨迹编号']) == set(conflict_table['轨迹编号']) == set(raw_table['轨迹编号'])
assert path_table['冲突接续数'].sum() == len(conflict_table)
assert path_table['GNSS原始行数'].sum() == len(raw_table)
assert path_table['城市缺失行数'].sum() + visit_table['原始记录数'].sum() == len(raw_table)
assert (path_table['冲突一级数'] + path_table['冲突多级数']).equals(path_table['冲突接续数'])
visit_lookup = visit_table.set_index(['轨迹编号', '访问序号'])
for _, edge in conflict_table.iterrows():
    for prefix in ['起点', '终点']:
        visit = visit_lookup.loc[(edge['轨迹编号'], int(edge[prefix + '访问序号']))]
        assert edge['边记录编号'] in visit['冲突' + prefix + '接续编号'].split('；')
        assert visit['城市'] == edge[prefix + '城市']
    assert pd.Timestamp(edge['终点进入时间']) < pd.Timestamp(edge['起点离开时间'])
for _, track in path_table.iterrows():
    visits = visit_table.loc[visit_table['轨迹编号'].eq(track['轨迹编号'])]
    raw_rows = raw_table.loc[raw_table['轨迹编号'].eq(track['轨迹编号'])]
    assert '→'.join(visits['城市']) == track['完整城市路径']
    assert len(raw_rows) == track['GNSS原始行数']
    assert raw_rows['轨迹内行序号'].tolist() == list(range(1, len(raw_rows) + 1))
    member_counts = raw_rows.groupby('访问序号').size()
    assert member_counts.to_dict() == visits.set_index('访问序号')['原始记录数'].to_dict()
    for _, visit in visits.iterrows():
        members = raw_rows.loc[raw_rows['访问序号'].eq(visit['访问序号'])]
        assert members['GNSS_center_location'].str.strip().eq(visit['城市']).all()
        assert pd.to_datetime(members['GNSS_start_time']).min() == pd.Timestamp(visit['进入时间'])
        assert pd.to_datetime(members['GNSS_end_time']).max() == pd.Timestamp(visit['离开时间'])
# 简单边界检查：后城提前开始才算冲突；时间刚好相等不算。
assert [(target - source).total_seconds() < 0 for source, target in [
    (pd.Timestamp('2023-05-01 10:00'), pd.Timestamp('2023-05-01 09:00')),
    (pd.Timestamp('2023-05-01 10:00'), pd.Timestamp('2023-05-01 10:00'))]] == [True, False]
print(f'检查通过：{len(path_table)}条完整轨迹、{len(conflict_table)}条冲突接续、'
      f'{len(visit_table)}次城市访问、{len(raw_table)}条不同GNSS原始行。')


检查通过：0条完整轨迹、0条冲突接续、0次城市访问、0条不同GNSS原始行。


## 6 输出

沿用同一个UTF-8文件Results/00_数据审计/冲突轨迹与GNSS.md，按年份、did分段。每条完整城市路径后紧接城市访问汇总表，每次连续同城访问一行。显示did、链段编号、城市、最早进入时间、最晚离开时间、GNSS行数，依次按访问顺序排列。

最早进入取该次访问的原始GNSS开始时间最小值，最晚离开取结束时间最大值。离开再返回同一城市分别列出，以便保留中间城市及检查时间冲突。GNSS行数不去重，逐段及全表核对数量；城市缺失而不能完整分组时停止，不悄悄漏行。

当前22段覆盖183次城市访问、1551条原始GNSS；只压缩显示，不删除原始行。旧四份CSV不改；修改前逐条GNSS版本已备份，普通重跑不自动归档。

In [5]:
report_lines, expected_rows = [], []
reported_gnss_rows = 0
assert int(path_table['城市缺失行数'].sum()) == 0, '冲突轨迹存在城市缺失，无法完整按城汇总，停止核对。'
for _, track in path_table.iterrows():
    track_visits = visit_table.loc[visit_table['轨迹编号'].eq(track['轨迹编号'])].sort_values('访问序号')
    assert len(track_visits) == int(track['城市访问数'])
    assert int(track_visits['原始记录数'].sum()) == int(track['GNSS原始行数'])
    assert '→'.join(track_visits['城市']) == track['完整城市路径']
    assert track_visits['城市'].ne(track_visits['城市'].shift()).all(), '连续同城未合并。'
    view_rows = track_visits[['农机编号','链段编号','城市','进入时间','离开时间','原始记录数']].rename(
        columns={'农机编号':'did','进入时间':'最早进入时间','离开时间':'最晚离开时间',
                 '原始记录数':'GNSS行数'}).astype('string')
    assert not view_rows.isna().any().any()
    assert pd.to_numeric(view_rows['GNSS行数']).gt(0).all()
    heading = (f"## {track['年份']}年 · did={track['农机编号']} · 链段{track['链段编号']}："
               + track['完整城市路径'])
    assert not any(character in heading for character in ['\r','\n'])
    if report_lines:
        report_lines.extend(['','---',''])
    report_lines.extend([heading,'',
        '| did | 链段编号 | 城市 | 最早进入时间 | 最晚离开时间 | GNSS行数 |',
        '| --- | --- | --- | --- | --- | --- |'])
    group_rows = list(view_rows.itertuples(index=False,name=None))
    for row in group_rows:
        assert all(not any(character in value for character in ['|','\r','\n']) for value in row)
        report_lines.append('| ' + ' | '.join(row) + ' |')
    expected_rows.extend(group_rows)
    reported_gnss_rows += int(track_visits['原始记录数'].sum())
if path_table.empty:
    report_lines = ['# 时间冲突检查结果','',
        '本次检查三年全部候选接续，包含一级、多级及起终点同城的记录。终点进入早于起点离开才算时间冲突；相等不算。','']
    report_lines += [f"{row['年份']}年：候选接续{row['候选接续数']:,}条，时间冲突{row['冲突接续数']}条，涉及轨迹{row['冲突轨迹数']}条。"
                     for row in year_checks]
    report_lines += ['','当前没有时间冲突，不制造虚构轨迹或城市汇总行。人工删除前的标记原件已统一归档。']
report_text = '\n'.join(report_lines) + '\n'
assert sum(line.startswith('## ') for line in report_lines) == len(path_table)
assert len(expected_rows) == len(visit_table)
assert reported_gnss_rows == len(raw_table)
for relative,digest in source_hashes.items():
    assert file_fingerprint(project_root/relative) == digest, '运行期间输入文件发生变化。'
result_dir.mkdir(parents=True,exist_ok=True)
report_path.write_text(report_text,encoding='utf-8')
saved_text = report_path.read_text(encoding='utf-8')
assert saved_text == report_text
saved_rows = [tuple(cell.strip() for cell in line.strip('|').split('|'))
              for line in saved_text.splitlines()
              if line.startswith('| ') and not line.startswith('| did |') and not line.startswith('| --- |')]
assert saved_rows == expected_rows, 'Markdown城市访问汇总内容或顺序不符。'

# 旧CSV不重写，只更新生成版本和“旧格式保留”的用途说明。
legacy_ids = {f'CONFLICT_TRAJECTORY_{number}' for number in range(1,5)}
legacy_mask = manifest_table['file_id'].isin(legacy_ids)
assert legacy_mask.sum() == 4 and (project_root/legacy_generator_relative).is_file()
for _,entry in manifest_table.loc[legacy_mask].iterrows():
    assert file_fingerprint(project_root/entry['relative_path']) == entry['sha256'], '旧CSV已变化，停止核对。'
updated_manifest = manifest_table.copy()
updated_manifest.loc[legacy_mask,'source_or_generator'] = legacy_generator_relative
updated_manifest.loc[legacy_mask,'purpose'] = '保留的旧格式冲突查看明细，不再随当前Notebook刷新，不作研究输入'
updated_manifest.loc[legacy_mask,'notes'] = '人工删除前旧版本；原行号只对应删除前GNSS；仅供历史追溯，不用于当前数据定位'
updated_manifest.loc[legacy_mask,'formal_status'] = '历史结果'
entry = {
    'file_id':'CONFLICT_TRAJECTORY_GROUPED_MD','relative_path':report_path.relative_to(project_root).as_posix(),
    'file_name':report_path.name,'file_role':'inspection_output','formal_status':'正式','read_only':'false',
    'year':'2021-2023','time_range':'各年05-01至06-25','data_grain':'按实际冲突农机年度轨迹分段，一行一次连续同城访问；无冲突时为0行',
    'row_count':str(len(visit_table)),'file_size_bytes':str(report_path.stat().st_size),
    'sha256':file_fingerprint(report_path),'encoding':'UTF-8','source_or_generator':notebook_relative,
    'upstream':'；'.join(source_hashes),'purpose':'先看冲突完整城市路径，再看按连续同城访问汇总的进入、离开和GNSS行数',
    'notes':f'本次{len(path_table)}条冲突轨迹、{len(visit_table)}次城市访问、{len(raw_table)}条GNSS；无冲突时只显示检查说明，不生成虚构记录'}
updated_manifest = pd.concat([updated_manifest.loc[~updated_manifest['file_id'].eq(entry['file_id'])],
                             pd.DataFrame([entry],columns=manifest_table.columns)],ignore_index=True)
assert not updated_manifest['relative_path'].duplicated().any() and not updated_manifest['file_id'].duplicated().any()
check_path = result_dir/'时间冲突检查.csv'
check_table = pd.DataFrame(year_checks)
check_table.to_csv(check_path,index=False,encoding='utf-8-sig')
check_entry = {column:'' for column in manifest_table.columns}
check_entry.update(file_id='TIME_CONFLICT_CHECK_CURRENT',
    relative_path=check_path.relative_to(project_root).as_posix(),file_name=check_path.name,
    file_role='quality_output',formal_status='正式',read_only='false',year='2021-2023',
    time_range='各年05-01至06-25',data_grain='一行一个年份的全部候选接续冲突检查',
    row_count=str(len(check_table)),file_size_bytes=str(check_path.stat().st_size),
    sha256=file_fingerprint(check_path),encoding='UTF-8-SIG',source_or_generator=notebook_relative,
    upstream='；'.join(source_hashes),purpose='检查当前第一步全部候选接续是否还有负时间间隔',
    notes='不限城市与边级；时间刚好相等不算；不依赖第二步；0表示实际没有冲突')
updated_manifest = pd.concat([updated_manifest.loc[~updated_manifest['file_id'].eq(check_entry['file_id'])],
    pd.DataFrame([check_entry],columns=manifest_table.columns)],ignore_index=True)
pd.testing.assert_frame_equal(pd.read_csv(check_path,encoding='utf-8-sig'),check_table)
updated_manifest.to_csv(manifest_path,index=False,encoding='utf-8-sig')
from IPython.display import Markdown
display(Markdown('\n'.join(report_lines[:12]) + '\n\n（Notebook仅预览，全部记录见输出Markdown文件。）'))
print(f'完成：{len(path_table)}条完整轨迹、{len(visit_table)}次城市访问，汇总覆盖{reported_gnss_rows}行GNSS；已写入{report_path.name}，旧CSV不改。')


# 时间冲突检查结果

本次检查三年全部候选接续，包含一级、多级及起终点同城的记录。终点进入早于起点离开才算时间冲突；相等不算。

2021年：候选接续110,458条，时间冲突0条，涉及轨迹0条。
2022年：候选接续274,939条，时间冲突0条，涉及轨迹0条。
2023年：候选接续500,068条，时间冲突0条，涉及轨迹0条。

当前没有时间冲突，不制造虚构轨迹或城市汇总行。人工删除前的标记原件已统一归档。

（Notebook仅预览，全部记录见输出Markdown文件。）

完成：0条完整轨迹、0次城市访问，汇总覆盖0行GNSS；已写入冲突轨迹与GNSS.md，旧CSV不改。
